In [1]:
# @title Setup (chạy ô này trước)
# Colab bắt đầu với một máy trống — clone repo và cài dependency.
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("Day21-Track3-Finetuning-Lab")
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")


GPU: Tesla T4


# Hai ví dụ FT thua trên regression cố định
Chỉ sinh lại 15 câu phổ thông để giải thích hồi quy và bổ sung ca thua thật. Không thay eval, train hoặc verdict. Run All trên T4; tải ZIP và giải nén vào repo. Đây là ca regression, không được gọi thành ca ticket target thua.


In [2]:
# Setup Colab web — repo của học viên
from pathlib import Path
import os,sys,subprocess
ROOT=Path("/content/Day21-Track3-Finetuning-Lab")
REPO="https://github.com/HuyHaiThanh/K4-Track3A-DAY21-NguyenVanHuy-2A202602428-Finetuning-Lab.git"
if not ROOT.exists():
    subprocess.run(["git","clone",REPO,str(ROOT)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(ROOT / "requirements.txt")],check=True)
os.chdir(ROOT)
import torch
assert torch.cuda.is_available(), "Chọn T4 GPU trước khi Run All."


In [3]:
# Tự khôi phục backup công khai — tài khoản khác không cần mount Drive.
from pathlib import PurePosixPath
from zipfile import ZipFile,is_zipfile
import shutil
BACKUP_FILE_ID = "1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p"
BACKUP_FOLDER_URL = "https://drive.google.com/drive/folders/18wN-UeRbCC6YQk1lYVV7v0HTdt8wrSOk"
BACKUP_SIZE = 481443440
required = ["results/verdict.json","results/baselines_frozen.json","data/split/train.jsonl",
            "adapters/correct/adapter_model.safetensors","adapters/correct/adapter_config.json"]
if not all((ROOT / name).is_file() for name in required):
    subprocess.run([sys.executable,"-m","pip","install","-q","gdown>=5,<6"],check=True)
    import gdown
    archive_path=Path("/content/lab21_shared_backup.zip")
    if not archive_path.is_file() or archive_path.stat().st_size != BACKUP_SIZE:
        downloaded=gdown.download(id=BACKUP_FILE_ID,output=str(archive_path),quiet=False)
        assert downloaded, "Không tải được ZIP: kiểm tra quyền Anyone with the link hoặc giới hạn tải của Drive."
    assert archive_path.stat().st_size == BACKUP_SIZE, "ZIP chưa tải đủ hoặc file Drive đã thay đổi."
    assert is_zipfile(archive_path), "Nội dung tải về không phải ZIP."
    with ZipFile(archive_path) as archive:
        assert archive.testzip() is None, "ZIP bị lỗi CRC."
        for entry in archive.infolist():
            parts=PurePosixPath(entry.filename).parts
            assert ".." not in parts and not PurePosixPath(entry.filename).is_absolute(), "ZIP có đường dẫn không an toàn."
            positions=[i for i,x in enumerate(parts) if x in ("results","adapters","data")]
            if not positions: continue
            relative=PurePosixPath(*parts[positions[0]:])
            allowed=relative.parts[0] == "results" or relative.parts[:2] in (("data","split"),("adapters","correct"))
            if not allowed or entry.is_dir(): continue
            assert (entry.external_attr >> 16) & 0o170000 != 0o120000, "ZIP có symlink."
            target=(ROOT / str(relative)).resolve()
            assert ROOT.resolve() in target.parents
            target.parent.mkdir(parents=True,exist_ok=True)
            with archive.open(entry) as source,target.open("wb") as dest:
                shutil.copyfileobj(source,dest)
assert all((ROOT / name).is_file() for name in required), "Backup thiếu artifact cần thiết."
print("Đã có baseline và adapter correct. Bước tiếp theo chỉ inference, không train lại.")


Downloading...
From (original): https://drive.google.com/uc?id=1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p
From (redirected): https://drive.google.com/uc?id=1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p&confirm=t&uuid=3edf884c-9074-4c5b-b32d-e2bc68296c17
To: /content/lab21_shared_backup.zip
100%|██████████| 481M/481M [00:07<00:00, 65.1MB/s]


Đã có baseline và adapter correct. Bước tiếp theo chỉ inference, không train lại.


In [4]:
import os,sys,subprocess,base64
(ROOT/'scripts/collect_regression_examples.py').write_bytes(base64.b64decode("aW1wb3J0IHBhdGhsaWIsc3lzLGpzb24saGFzaGxpYgpST09UPXBhdGhsaWIuUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0Kc3lzLnBhdGguaW5zZXJ0KDAsc3RyKFJPT1QvJ3NyYycpKQpmcm9tIGxhYmtpdCBpbXBvcnQgZ2VuZXJhdGUsZXZhbHVhdGUgYXMgZXYscmVwb3J0CmZyb20gbGFia2l0LmNvbmZpZyBpbXBvcnQgZ2V0X3RpZXIKZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKZnJvemVuX3BhdGg9Uk9PVC8ncmVzdWx0cy9iYXNlbGluZXNfZnJvemVuLmpzb24nCnZlcmRpY3RfcGF0aD1ST09ULydyZXN1bHRzL3ZlcmRpY3QuanNvbicKaGFzaGVzPXtzdHIocCk6aGFzaGxpYi5zaGEyNTYocC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpIGZvciBwIGluIFtmcm96ZW5fcGF0aCx2ZXJkaWN0X3BhdGhdfQpmcm96ZW49anNvbi5sb2Fkcyhmcm96ZW5fcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykpCnZlcmRpY3Q9anNvbi5sb2Fkcyh2ZXJkaWN0X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQp0aWVyPWdldF90aWVyKGZyb3plblsndGllciddKQphc3NlcnQgdGllci5tb2RlbF9pZD09ZnJvemVuWydtb2RlbCddIGFuZCBub3QgZnJvemVuLmdldCgnc21va2VfbW9kZScpCnJvd3M9W2pzb24ubG9hZHMobCkgZm9yIGwgaW4gKFJPT1QvJ2RhdGEvZXZhbF9yZWdyZXNzaW9uLmpzb25sJykucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKSBpZiBsXQphc3NlcnQgbGVuKHJvd3MpPT0xNQptb2RlbCx0b2s9Z2VuZXJhdGUubG9hZF9iYXNlKHRpZXIpCmJhc2UsXz1nZW5lcmF0ZS5nZW5lcmF0ZV9iYXRjaChtb2RlbCx0b2ssW3JbJ2luc3RydWN0aW9uJ10gZm9yIHIgaW4gcm93c10sc3lzdGVtPU5vbmUsbWF4X25ld190b2tlbnM9OTYsbGFiZWw9J3JlZ3Jlc3Npb24vYmFzZScpCmRlbCBtb2RlbDtnZW5lcmF0ZS5mcmVlX21lbW9yeSgpCm1vZGVsLHRvaz1nZW5lcmF0ZS5sb2FkX2Jhc2UodGllcikKbW9kZWw9UGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChtb2RlbCxzdHIoUk9PVC8nYWRhcHRlcnMvY29ycmVjdCcpKTttb2RlbC5ldmFsKCkKZnQsXz1nZW5lcmF0ZS5nZW5lcmF0ZV9iYXRjaChtb2RlbCx0b2ssW3JbJ2luc3RydWN0aW9uJ10gZm9yIHIgaW4gcm93c10sc3lzdGVtPU5vbmUsbWF4X25ld190b2tlbnM9OTYsbGFiZWw9J3JlZ3Jlc3Npb24vZnQnKQpkZWwgbW9kZWw7Z2VuZXJhdGUuZnJlZV9tZW1vcnkoKQpleGFtcGxlcz1bXQpmb3IgaSwocixiLHApIGluIGVudW1lcmF0ZSh6aXAocm93cyxiYXNlLGZ0KSk6CiBzYj1ldi5rZXl3b3JkX3JlY2FsbChiLHJbJ2tleXdvcmRzJ10pO3NmPWV2LmtleXdvcmRfcmVjYWxsKHAsclsna2V5d29yZHMnXSkKIGV4YW1wbGVzLmFwcGVuZCh7J2knOmksJ3F1ZXN0aW9uJzpyWydpbnN0cnVjdGlvbiddLCdrZXl3b3Jkcyc6clsna2V5d29yZHMnXSwncHJlZF9iJzpiLCdwcmVkX2Z0JzpwLCdzY29yZV9iJzpzYiwnc2NvcmVfZnQnOnNmLCdkZWx0YSc6c2Ytc2J9KQpibWVhbj1zdW0oclsnc2NvcmVfYiddIGZvciByIGluIGV4YW1wbGVzKS8xNQpmbWVhbj1zdW0oclsnc2NvcmVfZnQnXSBmb3IgciBpbiBleGFtcGxlcykvMTUKZXhwZWN0ZWQ9bmV4dChyWydyZWdyZXNzaW9uJ10gZm9yIHIgaW4gdmVyZGljdFsnY29tcGFyaXNvbiddIGlmICcoYyknIGluIHJbJ3J1biddKQptYXRjaGVkPWFicyhibWVhbi1mcm96ZW5bJ2Jhc2VsaW5lX2InXVsncmVncmVzc2lvbiddKTwxZS02IGFuZCBhYnMoZm1lYW4tZXhwZWN0ZWQpPDAuMDAwMQpsb3Npbmc9c29ydGVkKFtyIGZvciByIGluIGV4YW1wbGVzIGlmIHJbJ2RlbHRhJ108MF0sa2V5PWxhbWJkYSByOnJbJ2RlbHRhJ10pCnJlcG9ydC53cml0ZV9qc29uKGV4YW1wbGVzLCdyZWdyZXNzaW9uX2NvbXBhcmlzb24uanNvbicscmVzdWx0c19kaXI9Uk9PVC8ncmVzdWx0cycpCnJlcG9ydC53cml0ZV9qc29uKHsnbWF0Y2hlc19mcm96ZW5fc2NvcmVzJzptYXRjaGVkLCdiYXNlbGluZV9yZWdyZXNzaW9uJzpibWVhbiwnZnRfcmVncmVzc2lvbic6Zm1lYW4sJ2xvc3NfY291bnQnOmxlbihsb3NpbmcpfSwncmVncmVzc2lvbl9yZXByb2R1Y3Rpb25fY2hlY2suanNvbicscmVzdWx0c19kaXI9Uk9PVC8ncmVzdWx0cycpCnJlcG9ydC53cml0ZV9qc29uKGxvc2luZ1s6Ml0sJ3JlZ3Jlc3Npb25fc2VsZWN0ZWRfbG9zc2VzLmpzb24nLHJlc3VsdHNfZGlyPVJPT1QvJ3Jlc3VsdHMnKQpmb3IgcCxoIGluIGhhc2hlcy5pdGVtcygpOmFzc2VydCBoYXNobGliLnNoYTI1NihwYXRobGliLlBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKT09aApwcmludChmJ0Jhc2VsaW5lPXtibWVhbn07IEZUPXtmbWVhbn07IGto4bubcD17bWF0Y2hlZH07IGNhIHRodWE9e2xlbihsb3NpbmcpfScpCmFzc2VydCBtYXRjaGVkLCfEkGnhu4NtIGLhu5Ugc3VuZyBraMO0bmcga2jhu5twOyBraeG7g20gdHJhIG3DtGkgdHLGsOG7nW5nIHRyxrDhu5tjIGtoaSBkw7luZyB0cm9uZyByZXBvcnQuJwphc3NlcnQgbGVuKGxvc2luZyk+PTIsJ0tow7RuZyDEkeG7pyBoYWkgY2EgcmVncmVzc2lvbiB0aHVhIHRyw6puIHThuq1wIGPhu5EgxJHhu4tuaC4nCg=="))
os.environ['COMPUTE_TIER']='T4'
os.environ.pop('BASE_MODEL',None)
os.environ.pop('EVAL_LIMIT',None)
subprocess.run([sys.executable,'-u','scripts/collect_regression_examples.py'],cwd=ROOT,check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', 'scripts/collect_regression_examples.py'], returncode=0)

In [5]:
from zipfile import ZipFile,ZIP_DEFLATED
from google.colab import files
output=ROOT.parent/'lab21_regression_evidence.zip'
with ZipFile(output,'w',ZIP_DEFLATED) as z:
 for name in ['regression_comparison.json','regression_reproduction_check.json','regression_selected_losses.json']:
  z.write(ROOT/'results'/name,'results/'+name)
files.download(str(output))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>